## Project Overview: Logistics Connectivity

In modern agriculture, the transition from harvest to processing is a race against time and cost. This project focuses on the **spatial representation of agricultural flow**, mapping the transit corridors of three critical commodities: **Maize, Soybeans, and Wheat**. 

By utilizing dynamic vector paths, we can visualize the connectivity between production hubs and export terminals, providing a clear picture of regional logistics integration.

### Problem Statement
Fragmented logistics data often hides the complexity of multi-commodity transport. Understanding the "path of least resistance" and the overlap of different crop routes is essential for:
* **Infrastructure Planning:** Identifying high-traffic corridors that require maintenance.
* **Cost Mitigation:** Optimizing haulage distances to reduce fuel consumption.
* **Risk Assessment:** Visualizing dependency on specific transit bottlenecks.

### Technical Objectives
This analysis implements advanced line-string visualization techniques:
* **Dynamic Flow Mapping (AntPath):** Using animated polylines to indicate the direction and "pulse" of commodity movement.
* **Multivariate Color Coding:** Assigning specific hues to different crops to analyze route overlap and specialization.
* **Spatial Intersection:** Understanding how different supply chains converge at primary logistical nodes.

### Technical Architecture
To ensure scalability and performance, the project utilizes the following stack:

| Tool | Core Function | Application in Project |
|:---|:---|:---|
| **GeoPandas** | Spatial Vector Engine | Construction and manipulation of LineString geometries for transit routes. |
| **Folium (AntPath)** | Animation Engine | Deployment of non-static, animated polylines to represent flow direction. |
| **Branca** | HTML/CSS Styling | Advanced customization of interactive legends and dashboard components. |

---

## Network Acquisition and Topology Cleaning

In this stage, we utilize **OSMNX** to download the real-world road infrastructure from OpenStreetMap. A critical step in supply chain modeling is ensuring **network connectivity**; otherwise, routing algorithms would fail due to isolated road segments.

### Graph Extraction and Connectivity Audit
We extract a driving network centered on a strategic agricultural hub. To ensure a valid model, we perform a **Connected Components** analysis to retain only the largest interconnected subgraph, eliminating any "dead-end" segments that aren't physically linked to the main infrastructure.

In [3]:
import osmnx as ox
import networkx as nx
import folium
from folium.plugins import AntPath

# 1. Geographic Configuration (Agricultural Core Region)
# Centered near a strategic logistical node
center_coords = (-33.89, -60.57) 
print("Downloading road infrastructure from OpenStreetMap...")

# 2. Network Retrieval and Topology Refinement
# Downloads the road network within a 20km radius
G_raw = ox.graph_from_point(center_coords, dist=20000, network_type='drive')

# 3. Ensuring Graph Connectivity
# We convert to undirected to find connected components and extract the 
# Largest Strongly Connected Component (LSCC) to ensure routing feasibility.
G_weak = G_raw.to_undirected()
largest_component = max(nx.connected_components(G_weak), key=len)
G = G_raw.subgraph(largest_component).copy()

# 4. Critical Node Identification
# Locating the nearest network node to our primary processing facility (Plant)
target_plant_node = ox.nearest_nodes(G, center_coords[1], center_coords[0])

print(f"Network processing complete. Total nodes in analyzed subgraph: {len(G.nodes)}")

Network processing complete. Total nodes in analyzed subgraph: 4350


## Supply Node Configuration: Farm-to-Plant Logistics

In this section, we define the **Source Nodes** (Farms) that represent the origin of the agricultural supply chain. Each node is characterized by its geographic coordinates, current storage volume in metric tons (MT), and the specific commodity being transported.

### Commodity Segmentation
To visualize the multi-product flow, we categorized the supply into three strategic pillars:
* **Maize (Yellow):** High-volume grain transport.
* **Soybeans (Green):** Critical oilseed export commodity.
* **Wheat (Orange):** Essential cereal for regional milling.

In [4]:
# Production Node Dataset (Off-road Origins)
# This dataset represents the supply nodes for the multi-commodity logistics model.
# NOTE: Farm names (Spanish) are preserved for regional context.

farms_data = [
    {"name": "Finca El Sol", "coords": (-33.80, -60.48), "ton": 80, "commodity": "Maize", "color": "#f1c40f"},
    {"name": "Don Pedro", "coords": (-33.97, -60.42), "ton": 40, "commodity": "Soybeans", "color": "#2ecc71"},
    {"name": "La Posta", "coords": (-33.83, -60.78), "ton": 120, "commodity": "Wheat", "color": "#e67e22"},
    {"name": "El Trébol", "coords": (-34.05, -60.58), "ton": 65, "commodity": "Maize", "color": "#f1c40f"},
    {"name": "Santa Rita", "coords": (-33.75, -60.68), "ton": 90, "commodity": "Soybeans", "color": "#2ecc71"},
    {"name": "Campo Largo", "coords": (-33.88, -60.85), "ton": 110, "commodity": "Wheat", "color": "#e67e22"},
    {"name": "Las Lilas", "coords": (-34.08, -60.38), "ton": 35, "commodity": "Maize", "color": "#f1c40f"},
    {"name": "El Amanecer", "coords": (-33.72, -60.55), "ton": 55, "commodity": "Soybeans", "color": "#2ecc71"}
]

print(f"Successfully initialized {len(farms_data)} supply nodes across 3 commodity categories.")

Successfully initialized 8 supply nodes across 3 commodity categories.


## Initialize Map (High-Contrast Dark Mode for Vector Clarity)

In [5]:
m = folium.Map(location=center_coords, zoom_start=11, tiles="cartodb dark_matter")

# 1. Custom Dashboard Header (Floating UI)
header_html = '''
    <div style="position: fixed; top: 15px; left: 50%; transform: translateX(-50%); z-index:9999; 
                background-color: rgba(0,0,0,0.7); color: white; padding: 10px 25px; 
                border-radius: 20px; border: 1px solid #555; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">
        <h3 style="margin:0; font-size:18px;"><b>AGRO-LOGISTICS: HARVEST FLOW ANALYSIS</b></h3>
    </div>
    '''
m.get_root().html.add_child(folium.Element(header_html))

# 2. Dynamic Footer (Metadata & Attribution)
footer_html = '''
    <div style="position: fixed; bottom: 15px; left: 50%; transform: translateX(-50%); z-index:9999; 
                color: #888; font-size: 9px; font-family: 'Arial'; letter-spacing: 1px;">
        Author: @alexisdaraujoc | Data: OpenStreetMap | © 2026
    </div>
    '''
m.get_root().html.add_child(folium.Element(footer_html))

# 3. Layer Groups by Commodity Type
commodity_layers = {
    "Maize": folium.FeatureGroup(name='<span style="color:#f1c40f">●</span> Maize').add_to(m),
    "Soybeans": folium.FeatureGroup(name='<span style="color:#2ecc71">●</span> Soybeans').add_to(m),
    "Wheat": folium.FeatureGroup(name='<span style="color:#e67e22">●</span> Wheat').add_to(m)
}

## Routing Engine: Pathfinding and Kinetic Mapping (AntPath)

Calculating Dijkstra's shortest path between Farm Nodes and the Central Plant.

In [6]:
# 1. Calculating Dijkstra's shortest path between Farm Nodes and the Central Plant.

for farm in farms_data:
    try:
        # Snap farm coordinates to the nearest network node
        farm_node = ox.nearest_nodes(G, farm["coords"][1], farm["coords"][0])
        
        # Calculate optimal route based on distance (length)
        optimal_path = nx.shortest_path(G, farm_node, target_plant_node, weight='length')
        
        # Generate geometry coordinates: [Origin] -> [Road Segments]
        route_geometry = [list(farm["coords"])] + [[G.nodes[node]['y'], G.nodes[node]['x']] for node in optimal_path]
        
        # Deploy Animated Polyline (Weight proportional to Load Capacity)
        AntPath(
            locations=route_geometry,
            color=farm["color"],
            pulse_color='#ffffff',
            weight=farm["ton"]/8,  # Dynamic line weight based on cargo volume
            delay=1100,
            tooltip=f"<b>{farm['name']}</b><br>Commodity: {farm['commodity']}<br>Load: {farm['ton']} MT"
        ).add_to(commodity_layers[farm["commodity"]])
        
        # Source Marker (Origin Point)
        folium.Marker(
            location=farm["coords"],
            icon=folium.Icon(color='black', icon_color=farm["color"], icon='leaf', prefix='fa'),
            popup=farm["name"]
        ).add_to(commodity_layers[farm["commodity"]])
        
    except Exception as e:
        print(f"Error processing route for {farm['name']}: {e}")
        continue

Error processing route for Don Pedro: No path between 4052042735 and 8039791631.


## Logistical Infrastructure and Control Interface

In [8]:
from branca.element import Figure
import os

# 1 UI Sizing (800px vertical expansion)
fig = Figure(height=800)

# 2 Cleanup: Removing duplicate LayerControls before final render
for key in list(m._children.keys()):
    if 'layer_control' in key:
        del m._children[key]

# 3 Destination Node: Central Processing Plant
folium.Marker(
    location=center_coords,
    icon=folium.Icon(color='red', icon='industry', prefix='fa'),
    popup="CENTRAL PROCESSING PLANT"
).add_to(m)

# 4 Single Instance Layer Control
folium.LayerControl(collapsed=False).add_to(m)

# 5 Encapsulation
fig.add_child(m)

# 6 FORCED EXPORT SYSTEM
# We define the absolute path to avoid Windows permission issues
filename = "day2_lines_antpath.html"
current_dir = os.getcwd()
full_path = os.path.join(current_dir, filename)

try:
    # Force save
    m.save(full_path)
    
    # 7 Verification Check
    if os.path.exists(full_path):
        print(f"✅ SUCCESS: File generated")
    else:
        print("❌ ERROR: The file was not found after the save command.")
except Exception as e:
    print(f"❌ CRITICAL ERROR during save: {e}")

# 8 Final Render for Notebook
fig

✅ SUCCESS: File generated


## Strategic Findings and Analytical Conclusions

The implementation of a dynamic routing model for multi-commodity agricultural flows provides critical insights into the regional supply chain. By analyzing the animated transit corridors between the storage units (Farms) and the processing hub, several logistical patterns emerge.

### Key Analytical Insights
* **Corridor Pressure Analysis:** The use of **weighted polylines** (proportional to tonnage) reveals the "High-Load Corridors." Identifying these segments is vital for local infrastructure maintenance, as they bear the highest mechanical stress from heavy-duty grain transport.
* **Dynamic Flow Directionality:** Utilizing **AntPath** allows for the visualization of real-time "pulse" in the network. This differentiates between static infrastructure and active movement, providing a more intuitive understanding of harvest-time logistics.
* **Multi-Commodity Convergence:** The spatial intersection of Maize, Soybeans, and Wheat routes highlights potential bottlenecks. Strategic planning at these convergence points can significantly reduce transit latency and fuel consumption.

### Technical Summary
1.  **Network Fidelity:** Leveraging **OSMNX** ensured that the shortest path calculations were based on the actual topology of the driving network, providing a 1:1 reflection of field operations.
2.  **Scalable UI:** The integration of a custom CSS header and a high-contrast **Dark Matter** base map enhances the legibility of vector data, making the dashboard suitable for executive-level reporting.
3.  **Optimization Potential:** This model serves as a foundation for future **Last-Mile Optimization** and **Vehicle Routing Problem (VRP)** simulations within the regional agricultural sector.

---
*Developed by Alexis D. Araujo C. | Agricultural Engineer & Geospatial Data Scientist*